# BDC Satria Data 2026 — ConvNeXt V2-

- **Model:** `convnextv2_base.fcmae_ft_in22k_in1k` — **~89M params** (bukan Large 198M yang sudah kamu coba)
- **Resolusi:** 224
- **Environment:** Laptop pribadi, RTX 3050 (~4.3GB VRAM)

**Kenapa worth dicoba:** kamu sudah punya ConvNeXt V2-**Large** (198M) → **0.9720**. Tapi pola yang sangat konsisten di kasus kamu: **model lebih kecil sering menang dari yang lebih besar dalam 1 keluarga**:
- Swin-Base (88M, 0.9776) > semua model Large/Huge kamu
- ViT-B SigLIP (86M, 0.9863) > ViT-L SWAG (305M, 0.9770) > ViT-H SWAG (632M, ~0.9766)
- Dosen kamu: ConvNeXt-**Small** (0.9788) jadi single lane terbaiknya

Jadi ada alasan kuat menduga **ConvNeXt V2-Base (89M) bisa mengalahkan ConvNeXt V2-Large (198M)** milik kamu — sekaligus jadi kandidat CNN murni yang lebih baik untuk stacking.

**Config laptop:** `BATCH_SIZE=8` + `GRAD_ACCUM_STEPS=2` (efektif 16), `NUM_WORKERS=0`. Kalau OOM, turunkan `BATCH_SIZE` ke 4 & naikkan `GRAD_ACCUM_STEPS` ke 4.

**Konsisten dgn notebook lain:** `StratifiedShuffleSplit` 3-fold 90/10, class weight balanced, macro F1, tracking misclassified → `.xlsx` 2 sheet.

## 1. Setup & Konfigurasi

In [ ]:
from __future__ import annotations
import os
import json
import random
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler
from torchvision import transforms
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from PIL import Image

# timm untuk ConvNeXt V2 (tidak ada di torchvision)
try:
    import timm
except ImportError:
    raise ImportError("timm belum terinstall. Jalankan: pip install timm")

In [ ]:
# === PATH DATASET ===
DATA_ROOT = Path(r"D:\Downloads\BDC2026")
TRAIN_DIR = DATA_ROOT / "train"
TEST_DIR = DATA_ROOT / "test"
MODELS_DIR = DATA_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)
OUTPUT_DIR = DATA_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

CLASS_FOLDERS = ["0_Recyclable", "1_Electronic", "2_Organic"]
CLASS_NAMES = ["Recyclable", "Electronic", "Organic"]
NUM_CLASSES = len(CLASS_NAMES)

# === CONFIG TRAINING ===
MODEL_NAME = "convnextv2_base.fcmae_ft_in22k_in1k"   # BASE (89M), bukan Large (198M)
IMG_SIZE = 224
BATCH_SIZE = 8           # laptop VRAM 4.3GB
GRAD_ACCUM_STEPS = 2     # efektif batch = 16
NUM_EPOCHS = 15
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 4
N_FOLDS = 3
FOLD_VAL_FRAC = 0.10
SEED = 42
NUM_WORKERS = 0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed()
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"Model: {MODEL_NAME} @ res {IMG_SIZE} | effective batch {BATCH_SIZE * GRAD_ACCUM_STEPS}")

## 2. Index Dataset dari Folder
Sama seperti notebook ViT: **tidak** load semua gambar ke memory. Cukup kumpulkan daftar `(path, label_idx)` — gambar di-load lazy per-batch lewat `Dataset`.

In [ ]:
def index_dataset(train_dir: Path, class_folders: list[str]):
    records = []
    for label_idx, folder_name in enumerate(class_folders):
        folder = train_dir / folder_name
        if not folder.is_dir():
            raise FileNotFoundError(f"Folder tidak ditemukan: {folder}")
        exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
        files = [p for p in folder.iterdir() if p.suffix.lower() in exts]
        for p in files:
            records.append({"path": str(p), "label": label_idx, "class_name": CLASS_NAMES[label_idx]})
    return pd.DataFrame(records)

full_df = index_dataset(TRAIN_DIR, CLASS_FOLDERS)
print(f"Total citra ter-index: {len(full_df)}")
print(full_df["class_name"].value_counts())

## 3. Dataset & Transforms
Augmentasi hanya di train; val cuma resize + normalize (evaluasi jujur).

In [ ]:
class TrashDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            img = Image.open(row["path"]).convert("RGB")
        except Exception:
            img = Image.new("RGB", (IMG_SIZE, IMG_SIZE))
        if self.transform:
            img = self.transform(img)
        # kembalikan juga path biar bisa dilacak saat misclassified tracking
        return img, int(row["label"]), row["path"]


def get_transforms(train: bool = True):
    if train:
        return transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.RandomResizedCrop(IMG_SIZE, scale=(0.8, 1.0)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])
    return transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

# BDC Satria Data 2026 — ConvNeXt V2-**Base** (Laptop, VRAM ~4.3GB)

- **Model:** `convnextv2_base.fcmae_ft_in22k_in1k` — **~89M params** (bukan Large 198M yang sudah kamu coba)
- **Resolusi:** 224
- **Environment:** Laptop pribadi, RTX 3050 (~4.3GB VRAM)

**Kenapa worth dicoba:** kamu sudah punya ConvNeXt V2-**Large** (198M) → **0.9720**. Tapi pola yang sangat konsisten di kasus kamu: **model lebih kecil sering menang dari yang lebih besar dalam 1 keluarga**:
- Swin-Base (88M, 0.9776) > semua model Large/Huge kamu
- ViT-B SigLIP (86M, 0.9863) > ViT-L SWAG (305M, 0.9770) > ViT-H SWAG (632M, ~0.9766)
- Dosen kamu: ConvNeXt-**Small** (0.9788) jadi single lane terbaiknya

Jadi ada alasan kuat menduga **ConvNeXt V2-Base (89M) bisa mengalahkan ConvNeXt V2-Large (198M)** milik kamu — sekaligus jadi kandidat CNN murni yang lebih baik untuk stacking.

**Config laptop:** `BATCH_SIZE=8` + `GRAD_ACCUM_STEPS=2` (efektif 16), `NUM_WORKERS=0`. Kalau OOM, turunkan `BATCH_SIZE` ke 4 & naikkan `GRAD_ACCUM_STEPS` ke 4.

**Konsisten dgn notebook lain:** `StratifiedShuffleSplit` 3-fold 90/10, class weight balanced, macro F1, tracking misclassified → `.xlsx` 2 sheet.

In [ ]:
def build_model(num_classes: int):
    # num_classes=0 -> timm buang classifier head, kita ganti sendiri
    model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=num_classes)

    # Freeze semua dulu
    for p in model.parameters():
        p.requires_grad = False

    # Unfreeze hanya classifier head. Di ConvNeXt V2 timm, head ada di model.head.fc
    head = model.head
    for p in head.parameters():
        p.requires_grad = True

    return model.to(DEVICE)


# cek struktur head sekali (informational)
_tmp = build_model(NUM_CLASSES)
n_total = sum(p.numel() for p in _tmp.parameters())
n_train = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print(f"Total params: {n_total:,}")
print(f"Trainable params (head): {n_train:,}")
del _tmp
torch.cuda.empty_cache()

## 5. Train & Evaluate
`evaluate` mengembalikan probabilitas softmax + path tiap sampel, supaya bisa dipakai untuk tracking misclassified.

In [ ]:
scaler = GradScaler("cuda", enabled=(DEVICE == "cuda"))


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []
    optimizer.zero_grad()

    for step, (imgs, labels, _) in enumerate(loader):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast("cuda", enabled=(DEVICE == "cuda")):
            outputs = model(imgs)
            loss = criterion(outputs, labels) / GRAD_ACCUM_STEPS
        scaler.scale(loss).backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

        running_loss += loss.item() * GRAD_ACCUM_STEPS
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return running_loss / len(loader), macro_f1


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels, all_paths, all_probs = [], [], [], []
    for imgs, labels, paths in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with autocast("cuda", enabled=(DEVICE == "cuda")):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        running_loss += loss.item()

        probs = F.softmax(outputs.float(), dim=1)
        preds = probs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_paths.extend(paths)
        all_probs.extend(probs.cpu().numpy())

    macro_f1 = f1_score(all_labels, all_preds, average="macro")
    return running_loss / len(loader), macro_f1, np.array(all_labels), np.array(all_preds), all_paths, np.array(all_probs)

## 6. Cross Validation 3-Fold + Kumpulkan Misclassified

Tiap fold, setelah training selesai, model terbaik (macro F1 val tertinggi) di-evaluate ulang ke val set fold itu untuk mengambil prediksi final. Dua struktur dikumpulkan lintas fold:
- `detail_records` — tiap kejadian salah (untuk sheet **Detail**)
- `eval_counter` — berapa kali tiap gambar masuk val (untuk `total_evaluated` di sheet **Summary**)

In [ ]:
sss = StratifiedShuffleSplit(n_splits=N_FOLDS, test_size=FOLD_VAL_FRAC, random_state=SEED)

fold_results = []
detail_records = []                 # untuk sheet Detail
eval_counter = defaultdict(int)     # path -> berapa kali masuk val (total_evaluated)

for fold, (train_idx, val_idx) in enumerate(sss.split(full_df, full_df["label"]), start=1):
    print(f"\n{'='*64}\n---> FOLD {fold}/{N_FOLDS} <---\n{'='*64}")
    train_df = full_df.iloc[train_idx]
    val_df = full_df.iloc[val_idx]

    # class weight dari label training fold ini
    weights = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=train_df["label"].values)
    class_weights = torch.tensor(weights, dtype=torch.float32).to(DEVICE)
    print(f"  Class weights: {dict(zip(CLASS_NAMES, weights.round(3)))}")

    train_loader = DataLoader(TrashDataset(train_df, get_transforms(True)),
                              batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
    val_loader = DataLoader(TrashDataset(val_df, get_transforms(False)),
                            batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

    model = build_model(NUM_CLASSES)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                                  lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

    best_f1 = 0.0
    epochs_no_improve = 0
    best_path = MODELS_DIR / f"convnextv2_base_fold{fold}.pth"

    for epoch in range(1, NUM_EPOCHS + 1):
        tr_loss, tr_f1 = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_f1, _, _, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step(val_f1)

        marker = ""
        if val_f1 > best_f1:
            best_f1 = val_f1
            epochs_no_improve = 0
            torch.save(model.state_dict(), best_path)
            marker = " [*] best"
        else:
            epochs_no_improve += 1

        print(f"  Epoch {epoch:2d}/{NUM_EPOCHS} | train_loss {tr_loss:.4f} F1 {tr_f1:.4f} | "
              f"val_loss {val_loss:.4f} F1 {val_f1:.4f}{marker}")

        if epochs_no_improve >= PATIENCE:
            print(f"  Early stopping (tidak ada peningkatan {PATIENCE} epoch).")
            break

    # Evaluate ulang pakai model terbaik fold ini utk ambil prediksi final -> tracking misclassified
    model.load_state_dict(torch.load(best_path, weights_only=True))
    _, final_f1, y_true, y_pred, paths, probs = evaluate(model, val_loader, criterion)
    print(f"  >> Fold {fold} best macro F1: {best_f1:.4f} | re-eval F1: {final_f1:.4f}")

    fold_results.append({"fold": fold, "best_macro_f1": best_f1})

    # Catat evaluasi & kesalahan
    for i, path in enumerate(paths):
        eval_counter[path] += 1
        if y_true[i] != y_pred[i]:
            p = probs[i]
            detail_records.append({
                "fold": fold,
                "filepath": path,
                "true_label": CLASS_NAMES[y_true[i]],
                "predicted_label": CLASS_NAMES[y_pred[i]],
                "confidence_pct": round(float(p[y_pred[i]]) * 100, 2),
                "true_label_confidence_pct": round(float(p[y_true[i]]) * 100, 2),
                "prob_recyclable_pct": round(float(p[0]) * 100, 2),
                "prob_electronic_pct": round(float(p[1]) * 100, 2),
                "prob_organic_pct": round(float(p[2]) * 100, 2),
            })

    del model, train_loader, val_loader
    torch.cuda.empty_cache()

## 7. Ringkasan Hasil 3-Fold

In [ ]:
f1s = [r["best_macro_f1"] for r in fold_results]
print("Ringkasan 3-Fold CV (macro F1):")
for r in fold_results:
    print(f"  Fold {r['fold']}: {r['best_macro_f1']:.4f}")
print(f"\nRata-rata macro F1: {np.mean(f1s):.4f} (+/- {np.std(f1s):.4f})")

print("\nPembanding (punya kamu):")
print("  DenseNet-201         : 0.9403")
print("  DeiT-Base            : 0.9513")
print("  PVTv2-B5             : 0.9535")
print("  RegNetY-16GF         : 0.9599")
print("  ViT-B/16 SWAG        : 0.9636")
print("  EVA-02 Base          : 0.9646")
print("  BEiT-Base            : 0.9695")
print("  CoAtNet-2            : 0.9698")
print("  ConvNeXt V2-LARGE    : 0.9720  <- versi Large, apakah Base bisa kalahkan ini?")
print("  ViT-L/16 SWAG        : 0.9770")
print("  Swin-Base V2         : 0.9776")
print("  ViT-B/16 SigLIP1     : 0.9863  <- juara sejauh ini")

pd.DataFrame(fold_results).to_csv(OUTPUT_DIR / "convnextv2_base_fold_results.csv", index=False)

## 8. Bangun Laporan Misclassified (`.xlsx`, 2 sheet)

- **Detail**: langsung dari `detail_records`.
- **Summary**: agregasi per `filepath` unik. `folds_wrong` = daftar fold tempat gambar salah; `total_evaluated` dari `eval_counter`; `error_rate = total_wrong / total_evaluated`. Diurutkan `error_rate` lalu `total_wrong` menurun → gambar paling problematik di atas.

In [ ]:
detail_df = pd.DataFrame(detail_records, columns=[
    "fold", "filepath", "true_label", "predicted_label",
    "confidence_pct", "true_label_confidence_pct",
    "prob_recyclable_pct", "prob_electronic_pct", "prob_organic_pct",
])

# ---- Summary: agregasi per filepath unik ----
summary_rows = []
if len(detail_df) > 0:
    grouped = detail_df.groupby("filepath")
    for path, g in grouped:
        folds_wrong = sorted(g["fold"].unique().tolist())
        summary_rows.append({
            "filepath": path,
            "true_label": g["true_label"].iloc[0],   # true_label konsisten utk 1 gambar
            "folds_wrong": ",".join(map(str, folds_wrong)),
            "total_wrong": len(g),
            "total_evaluated": eval_counter[path],
            "error_rate": round(len(g) / eval_counter[path], 3) if eval_counter[path] else None,
            "avg_confidence_pct": round(g["confidence_pct"].mean(), 2),
            "avg_true_label_confidence_pct": round(g["true_label_confidence_pct"].mean(), 2),
            "avg_prob_recyclable_pct": round(g["prob_recyclable_pct"].mean(), 2),
            "avg_prob_electronic_pct": round(g["prob_electronic_pct"].mean(), 2),
            "avg_prob_organic_pct": round(g["prob_organic_pct"].mean(), 2),
        })

summary_df = pd.DataFrame(summary_rows, columns=[
    "filepath", "true_label", "folds_wrong", "total_wrong", "total_evaluated", "error_rate",
    "avg_confidence_pct", "avg_true_label_confidence_pct",
    "avg_prob_recyclable_pct", "avg_prob_electronic_pct", "avg_prob_organic_pct",
])
if len(summary_df) > 0:
    summary_df = summary_df.sort_values(["error_rate", "total_wrong"], ascending=False).reset_index(drop=True)

report_path = OUTPUT_DIR / "misclassified_report_convnextv2_base.xlsx"
with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
    detail_df.to_excel(writer, sheet_name="Detail", index=False)
    summary_df.to_excel(writer, sheet_name="Summary", index=False)

print(f"[SAVED] {report_path}")
print(f"  Detail  : {len(detail_df)} baris (total kejadian salah lintas fold)")
print(f"  Summary : {len(summary_df)} gambar unik yang pernah salah")
if len(summary_df) > 0:
    print("\n5 gambar paling problematik:")
    print(summary_df.head(5).to_string(index=False))

# BDC Satria Data 2026 — ConvNeXt V2-**Base** (Laptop, VRAM ~4.3GB)

- **Model:** `convnextv2_base.fcmae_ft_in22k_in1k` — **~89M params** (bukan Large 198M yang sudah kamu coba)
- **Resolusi:** 224
- **Environment:** Laptop pribadi, RTX 3050 (~4.3GB VRAM)

**Kenapa worth dicoba:** kamu sudah punya ConvNeXt V2-**Large** (198M) → **0.9720**. Tapi pola yang sangat konsisten di kasus kamu: **model lebih kecil sering menang dari yang lebih besar dalam 1 keluarga**:
- Swin-Base (88M, 0.9776) > semua model Large/Huge kamu
- ViT-B SigLIP (86M, 0.9863) > ViT-L SWAG (305M, 0.9770) > ViT-H SWAG (632M, ~0.9766)
- Dosen kamu: ConvNeXt-**Small** (0.9788) jadi single lane terbaiknya

Jadi ada alasan kuat menduga **ConvNeXt V2-Base (89M) bisa mengalahkan ConvNeXt V2-Large (198M)** milik kamu — sekaligus jadi kandidat CNN murni yang lebih baik untuk stacking.

**Config laptop:** `BATCH_SIZE=8` + `GRAD_ACCUM_STEPS=2` (efektif 16), `NUM_WORKERS=0`. Kalau OOM, turunkan `BATCH_SIZE` ke 4 & naikkan `GRAD_ACCUM_STEPS` ke 4.

**Konsisten dgn notebook lain:** `StratifiedShuffleSplit` 3-fold 90/10, class weight balanced, macro F1, tracking misclassified → `.xlsx` 2 sheet.